In [ ]:
import pandas as pd
import requests
import json
import time
import os
import time
import re

In [28]:
from openai import OpenAI


In [ ]:
client = OpenAI(
    api_key="Enter Your API Key Here",
    base_url="https://api.kluster.ai/v1"
)

In [30]:
df = pd.read_csv("R1_input_data.csv")
df

,input
0,Dice is the leading career destination for tec...
1,"In a world of possibilities, pursue one with e..."
2,About the job\nAbout Rocket Lawyer\n\nWe belie...
3,About the job\nThis role is with Maximus. WayU...
4,About the job\nXometry (NASDAQ: XMTR) powers t...
...,...
1020,Dimensional is a privately owned global invest...
1021,Sr OR & Advanced Analytics Specialist I (Data ...
1022,"**Portfolio Manager, Information Capital and D..."
1023,"As a Data Architect, your role will be to tran..."


In [31]:
# df = df[:4]


In [32]:
df

,input
0,Dice is the leading career destination for tec...
1,"In a world of possibilities, pursue one with e..."
2,About the job\nAbout Rocket Lawyer\n\nWe belie...
3,About the job\nThis role is with Maximus. WayU...
4,About the job\nXometry (NASDAQ: XMTR) powers t...
...,...
1020,Dimensional is a privately owned global invest...
1021,Sr OR & Advanced Analytics Specialist I (Data ...
1022,"**Portfolio Manager, Information Capital and D..."
1023,"As a Data Architect, your role will be to tran..."


In [33]:
def reasoning_prompt(job_description):
    return f"""
You are an experienced skill extraction model. Your task is to extract skills from the given job description using a 4-step reasoning process. Be concise and precise in your reasoning.

Step 1: Understand the Role and Context
Describe how the job title fits into the company’s structure or industry. Consider the function, domain, and purpose of the role.

Step 2: Extract Explicit Skills
List all tools, technologies, certifications, and named skills that are clearly and directly mentioned in the job description.
For each skill, provide a short reason explaining why it was extracted.

Step 3: Infer Implicit Skills
Using the context from Step 1 and the explicit skills from Step 2, infer additional skills that are not directly stated but are clearly implied by the responsibilities or expectations.
For each inferred skill, provide a short reason tied to the job description or context.

Step 4: Thinking Log
show your detailed reasoning process for each step above. This will be used to help train smaller models to mimic your thinking.

Output format:
## Thinking
step 1: ...
step 2: ...
step 3: ...

## Skills
skill 1(implicit): reason 1
skill 2(explicit): reason 2
...

Begin analysis using the job description below:

\"\"\"
{job_description}
\"\"\"
"""

In [34]:
import re

def clean_output(output_text):
    """
    Removes the <think>...</think> block from the output string.
    Returns the cleaned JSON string.
    """
    # Regex pattern to match <think> ... </think> including newlines
    think_pattern = re.compile(r"<think>.*?</think>", re.DOTALL)

    # Remove the think block
    cleaned_text = re.sub(think_pattern, "", output_text)

    # Strip whitespace
    return cleaned_text.strip()

In [ ]:
output_csv = "reasoning_outputs.csv"

start_idx = 592
calls = 0

if not os.path.isfile(output_csv):
    pd.DataFrame(columns=["input", "output"]).to_csv(output_csv, index=False)

for idx in range(start_idx, len(df)):
    job_description = df.loc[idx, "input"]
    prompt = reasoning_prompt(job_description)

    try:
        completion = client.chat.completions.create(
            model="deepseek-ai/DeepSeek-R1",
            max_completion_tokens=4000,
            temperature=0.6,
            top_p=1,
            messages=[
                {
                    "role": "system",
                    "content": "You are an experienced skill extraction model.",
                },
                {"role": "user", "content": prompt},
            ],
        )

        raw_output = completion.choices[0].message.content
        cleaned_output = clean_output(raw_output)

        current_row = pd.DataFrame({
            "input": [job_description],
            "output": [cleaned_output]
        })
        current_row.to_csv(output_csv, mode='a', index=False, header=False)

        print(f"Saved output for job description {idx+1}")

        if (idx + 1) % 25 == 0:
            checkpoint_df = pd.read_csv(output_csv)
            checkpoint_filename = f"checkpoint_{idx + 1}.csv"
            checkpoint_df.to_csv(checkpoint_filename, index=False)
            print(f"Checkpoint saved: {checkpoint_filename}")
            print("Sleeping for 60s to respect rate limits")
            time.sleep(60)
    except Exception as e:
        print(f"Error at index {idx}: {e}")


In [ ]:
results = pd.read_csv("reasoning_outputs.csv")

In [ ]:
results

In [ ]:
def parse_thinking(text):
    thinking = {}

    step_pattern = re.compile(
        r"(?:\*\*)?(step\s*\d+):\s*(?:\*\*)?(.*?)(?:\*\*)?\s*\n([\s\S]*?)(?=(?:\*\*)?step\s*\d+:|---|$)",
        re.IGNORECASE,
    )

    for match in step_pattern.finditer(text):
        step_num_raw, step_title, content = match.groups()
        step_num = step_num_raw.lower().replace(" ", "_")  # e.g., "step_1"
        step_title = step_title.strip()
        data = {}

        # Split content into lines, ignore empty lines
        lines = [line.strip("- ").strip() for line in content.strip().split("\n") if line.strip()]

        for line in lines:
            # Look for key: value lines with optional bold keys
            m = re.match(r"\*\*(.*?)\*\*:\s*(.*)", line)
            if m:
                key = m.group(1).strip().lower().replace(" ", "_")
                val = m.group(2).strip()
                if "," in val and not val.startswith("("):
                    val = [v.strip() for v in val.split(",")]
                data[key] = val
            else:
                # Accumulate lines without key under 'details'
                data.setdefault("details", []).append(line)

        if "details" in data:
            data["details"] = " ".join(data["details"])

        thinking[step_num] = {"title": step_title, **data}

    return thinking


def parse_explicit_implicit_skills(text):
    skills = {"explicit_skills": [], "implicit_skills": []}

    # Find explicit skills block
    explicit_block = re.search(r"\*\*Explicit Skills\*\*(.*?)(?=\*\*Implicit Skills\*\*|$)", text, re.S | re.I)
    if explicit_block:
        explicit_lines = explicit_block.group(1).strip().split("\n")
        for line in explicit_lines:
            m = re.match(r"- \*\*(.*?)\*\*.*?: (.*)", line.strip())
            if m:
                skill = m.group(1).strip()
                reason = m.group(2).strip()
                skills["explicit_skills"].append({"skill": skill, "reason": reason})

    # Find implicit skills block
    implicit_block = re.search(r"\*\*Implicit Skills\*\*(.*)", text, re.S | re.I)
    if implicit_block:
        implicit_lines = implicit_block.group(1).strip().split("\n")
        for line in implicit_lines:
            m = re.match(r"- \*\*(.*?)\*\*.*?: (.*)", line.strip())
            if m:
                skill = m.group(1).strip()
                reason = m.group(2).strip()
                skills["implicit_skills"].append({"skill": skill, "reason": reason})

    return skills


def parse_numbered_skills(text):
    skills = {"explicit_skills": [], "implicit_skills": []}

    # Match numbered skills like:
    # 1. **Skill (explicit)**: Reason
    # 2. **Skill (implicit)**: Reason
    pattern = re.compile(r"\d+\.\s*\*\*(.*?)\s*\((explicit|implicit)\)\*\*:\s*(.*)", re.I)
    for skill, typ, reason in pattern.findall(text):
        typ = typ.lower()
        skills[f"{typ}_skills"].append({"skill": skill.strip(), "reason": reason.strip()})

    return skills


def parse_skills(text):
    # Try explicit/implicit section first
    skills = parse_explicit_implicit_skills(text)

    # If no skills found, try numbered list format
    if not skills["explicit_skills"] and not skills["implicit_skills"]:
        skills = parse_numbered_skills(text)

    return skills


def parse_full_text(text):
    thinking_section = parse_thinking(text)
    skills_section = parse_skills(text)

    return {
        "thinking": thinking_section,
        "skills": skills_section
    }


In [ ]:
all_outputs = []

for idx, row in results.iterrows():
    job_description = row['input']
    text = row['output']

    # Use the combined parser function that includes thinking + skills parsing
    parsed = parse_full_text(text)

    # Convert the parsed dict to JSON string
    output_json = json.dumps([parsed], ensure_ascii=False)

    all_outputs.append({
        "input": job_description,
        "output": output_json
    })

# Convert list of dicts to DataFrame and save as CSV
df_out = pd.DataFrame(all_outputs)
df_out.to_csv("pre-processed_reasoning.csv", index=False, encoding='utf-8')
